In [167]:
import os
from pathlib import Path

import re
import geopandas as gpd
import pandas as pd

In [168]:
data_path = Path(os.environ["DATA_PATH"])
generated_path = data_path / "generated"
scripts_path = Path("./scripts")

In [169]:
df_orig = (
    pd.read_parquet(generated_path / "registro.parquet")
    .assign(fraccionamiento=lambda df: df["fraccionamiento"].where(lambda x: ~x.str.startswith("privadas condesa"), lambda x: x + " (" + df["privada"].str.casefold() + ")"))
)

In [175]:
def fix_condesa_col(x: str) -> str:
    if not x.startswith("privadas condesa"):
        return x

    res = re.search(r"privadas condesa seccion (.+)", x)
    if res is None:
        err = f"Unexpected format for {x}"
        raise ValueError(err)
    name = res.group(1)
    return f"privadas condesa ({name})"


df_col = (
    gpd.read_file(data_path / "initial" / "lim_cols_cp")
    .dropna(subset=["COLONIAS"])
    .assign(
        COLONIAS=lambda df: (
            df["COLONIAS"]
            .str.casefold()
            .str.replace(r"fracc\.?", "", regex=True)
            .str.replace("fraccionamiento", "")
            .str.strip()
            .replace(
                {
                    "quinta granada": pd.NA,
                    "valle de puebla": pd.NA,  # Replace with NA to fill with specific section
                    "la condesa": pd.NA,
                    "victoria residencial": pd.NA,
                },
            )
            .where(lambda x: ~x.isna(), df["Col_Secc"].str.casefold())
            .replace(
                {
                    "des. hab. privada campestre": "privadas campestre",
                    "camino del sol": "caminos del sol",
                    "balbuena condominios": "colonia balbuena",
                    "balbuena": "colonia balbuena",
                    "corceles": "corceles residencial",
                    "la condesa seccion fontalba": "fontalba residencial",
                    "valle de puebla quinta etapa": "valle de puebla 5ta secc.",
                    "quinta granada primera etapa": "quinta granada",
                    "quinta granada tercera etapa": "quinta granada 3",
                    "desarrollo habitacional residencial natura": "residencial natura",
                },
            )
            .str.replace("la condesa", "privadas condesa")
            .str.replace("seccio0n", "seccion")
            .str.replace("leganes", "leganés")
            .apply(fix_condesa_col)
            .replace({"fracc. victoria residencial segunda seccion": "privadas condesa (victoria segunda seccion)"})
        ),
    )
    [["COLONIAS", "geometry"]]
    .dissolve(by="COLONIAS")
)

In [179]:
df_mesh = gpd.read_file(generated_path / "mesh_accessibility.gpkg")

In [221]:
import requests
import json

response = requests.post(
    "http://127.0.0.1:8000/tree_coverage/geojson",
    json=json.loads(df_mesh.reset_index(names="CVEGEO").to_json()),
)

In [189]:
accessibility = (
    gpd.GeoDataFrame(
        df_col[["geometry"]]
        .to_crs("EPSG:6372")
        .sjoin(df_mesh[["geometry", "accessibility_score"]], how="left")
        .groupby("COLONIAS")
        .agg({"accessibility_score": "mean", "geometry": "first"}),
        crs=df_mesh.crs,
    )
    .rename(columns={"accessibility_score": "accessibility"})["accessibility"]
)

df_col = df_col.assign(accessibility=accessibility)

In [192]:
df_col

,geometry,accessibility
COLONIAS,,
18 de marzo,"POLYGON ((648559.899 3610456.764, 648521.344 3...",0.324100
27 de enero magisterial,"POLYGON ((656702.801 3608485.795, 656696.047 3...",0.119727
27 de septiembre,"POLYGON ((640189.487 3611588.872, 640195.276 3...",0.457595
5 de julio,"POLYGON ((646687.575 3607618.83, 646663.819 36...",0.147189
adara,"POLYGON ((651295.988 3610119.261, 651334.76 36...",0.395900
...,...,...
xochimilco,"POLYGON ((646643.064 3608831.853, 646685.374 3...",0.244676
zacatecas,"POLYGON ((643978.725 3611923.184, 643837.276 3...",0.470520
zona industrial,"MULTIPOLYGON (((646095.194 3609852.355, 646339...",0.427008
